# Rerun the prepared report and inspect text reuse

First complete [02](02_aapl_order_activity.ipynb). This notebook reads its immutable report definition and runs that definition again against the same source. It uses no supervisor or study author. Matching facts, instructions and model settings should reuse persisted text; provider traces, rather than matching text alone, establish zero new inference.

In [ ]:
from report_helpers import connect, task, native_operation, execute, read, dashboard
from epoch_sdk.research import ResearchCompleted

epoch = connect()
review = task('aapl_order_activity', attempt='demo-01')

## 1. Read the prepared report

This reads the completed operation from notebook 02. It never starts that notebook automatically.

In [ ]:
source = await epoch.research_status(operation_id=review.operation_id)
assert isinstance(source, ResearchCompleted), 'Complete notebook 02 first.'
original = await native_operation(epoch, source)
print('Original report execution:', original.execution_id)

## 2. Execute

A different operation ID creates a new report execution with the same definition, source, tables and parameters. Reusing this ID reconnects to that execution. This is a direct Backend operation; it does not run the original backtest.

In [ ]:
rerun = await execute(epoch, original, operation_id=review.operation_id + '-cache-01')
assert rerun.execution_id != original.execution_id
answer = await read(epoch, rerun, show=True)
dashboard_url = dashboard(rerun)

## 3. Compare the saved outputs

Read all table rows and compare the exact retained text and provenance. The renderer is the same one used by the ordinary SDK session. A policy/facts change is a different cache input and must not be reported as a hit.

In [ ]:
base = await read(epoch, source)
assert answer.tables == base.tables and answer.text == base.text
assert source.response.extension is not None and answer.reference.extension is not None
assert answer.reference.extension.source_execution_id == source.response.extension.source_execution_id
assert answer.reference.extension.commentary == source.response.extension.commentary
assert answer.reference.response_id != base.reference.response_id
reconnected = await epoch.research_status(operation_id=review.operation_id)
assert isinstance(reconnected, ResearchCompleted) and reconnected.response == source.response
answer

## 4. Close this client

Keep both operation IDs for later reads. The server retains the source, report and answers.

In [ ]:
await epoch.aclose()